In [ ]:
import torch

# 행은 사용자 idx를 의미하고
# 열은 아이템 idx를 의미함
ratings = torch.tensor([
    [1, 1, 0, 1, 0, 0],
    [0, 0, 1, 1, 1, 0],
    [1, 0, 0, 0, 1, 1],
], dtype=torch.float32) 
# float32로 놓는 이유는 이후에 loss 계산에 쓰이기 때문임 (labels과 모델의 예측 값과 비교하게 됨)

num_users, num_items = ratings.shape

In [ ]:
# 사용자들이 선택한 값들 추출하기
# bool 텐서를 이용해서 True인 인덱스들의 위치를 뽑아주기 (이 메서드에서는 False == 0으로 인식하기 때문)
positive_pairs = torch.nonzero(ratings == 1)

# 결과적으로 선택한 pos [user_idx, item_idx]를 뽑게됨
print(positive_pairs)

tensor([[0, 0],
        [0, 1],
        [0, 3],
        [1, 2],
        [1, 3],
        [1, 4],
        [2, 0],
        [2, 4],
        [2, 5]])


In [5]:
# postive_pairs에서 마지막 값만 뽑아보기
test_items = {}

for user_id in range(num_users):
  # 사용자 모든 id 뽑기
  # 이것도 boolmask를 만들어서 인덱스 위치 뽑아주기
  positive_items = torch.where(ratings[user_id] == 1)[0]

  test_items[user_id] = positive_items[-1].item()  # -> {user_id: item_idx} 로 이루어지게 됨.
  # 나중에 위의 값을 model(user_id, item_ids)를 통해 top_k를 구한 뒤, item_idx가 몇번째인지 구하기 위해 사용할 예정임

In [6]:
# 학습할 선택들만 뽑기 (test_items는 제외하기)
train_positive = []

for user_id in range(num_users):
  # 사용자 idx에 대한 `1` 인덱스들 구해주기
  positive_items = torch.where(ratings[user_id]==1)[0]

  # 인덱스들 중에서 이미 test_items에 들어있는 값은 제외해주기
  for item_id in positive_items:
    item_id = item_id.item()

    if item_id == test_items[user_id]:
      continue

    # 인덱스를 튜플로 넣어놓기
    train_positive.append((user_id, item_id))

print(train_positive)

[(0, 0), (0, 1), (1, 2), (1, 3), (2, 0), (2, 4)]


In [ ]:
# Negative Sampling
num_negatives = 2

user_id = 0

# 진짜 원본 배열
negative_candidates = torch.where(ratings[user_id] == 0)[0]

# 뽑을 negative_candidates의 인덱스 뽑아놓기
# 뽑을 인덱스 배열
indices = torch.randperm(len(negative_candidates))[:num_negatives]

# 뽑인 인덱스 배열
sampled_negatives = negative_candidates[indices]

print(sampled_negatives)

tensor([4, 5])


In [12]:
users = []
items = []
labels = []

num_negatives = 2

# 한번의 positive_sample마다 두번의 negative_sample을 추가해주기
for user_id, positive_item in train_positive:
  # positive sample 1개 추가해주기
  users.append(user_id)
  items.append(positive_item)
  labels.append(1.0)

  # 사용자가 선택하지 않은 아이템
  negative_candidates = torch.where(ratings[user_id] == 0)[0]

  # 랜덤하게 negative 추출
  # 사용자의 선택중에 임의의 num_negatives 개수만큼의 음성 데이터를 뽑은 후
  # 그중엥서 2(=num_negatives)개 뽑아주기
  indices = torch.randperm(len(negative_candidates))[:num_negatives]
  sampled_negatives = negative_candidates[indices]

  # 모두 [user_idx, neg_item_idx]로 추가해주기
  for negative_item in sampled_negatives:
    users.append(user_id)
    items.append(negative_item.item())
    # 음성
    labels.append(0.0)

In [17]:
# torch 타입으로 변경해주기
users = torch.tensor(users, dtype=torch.long)
items = torch.tensor(items, dtype=torch.long)
# 실수로 변경해주기
labels = torch.tensor(labels, dtype=torch.float32)

In [14]:
from torch import nn

class Recommender(nn.Module):

  def __init__(self, num_users, num_items, embedding_dim=4):
    super().__init__()

    self.user_embedding = nn.Embedding(num_users, embedding_dim)
    self.item_embedding = nn.Embedding(num_items, embedding_dim)

  def forward(self, user_ids, item_ids):
    user_vec = self.user_embedding(user_ids)
    item_vec = self.item_embedding(item_ids)

    return (user_vec * item_vec).sum(dim=1)


In [ ]:
model = Recommender(num_users, num_items)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(200):
  optimizer.zero_grad()

  # 뽑아놓은 훈련 유저셋과 아이템 데이터셋을 이용하여 점수 뽑아보기
  scores = model(users, items)

  # 손실값 구하기 + DAG 만들어주기
  loss = criterion(scores, labels)

  loss.backward()
  optimizer.step()

  if epoch % 20 == 0:
    print(epoch, loss.item())
# loss가 줄어드는 모습을 확인 가능함

0 0.5548655986785889
20 0.3379625678062439
40 0.1956055909395218
60 0.10918012261390686
80 0.06286191195249557
100 0.03927375376224518
120 0.02667412720620632
140 0.01933102495968342
160 0.014693268574774265
180 0.011572305113077164


In [20]:
with torch.no_grad():
  scores = model(users, items)
  print(scores)
  probs = torch.sigmoid(scores)
  print(probs)

tensor([ 4.7881, -4.5053, -7.5256,  4.1779, -7.5256, -3.9166,  5.0651, -5.0326,
        -4.6887,  6.1005, -4.6887, -5.0326,  3.9960, -4.8477, -4.6331,  4.0673,
        -4.6331, -4.4955])
tensor([9.9174e-01, 1.0929e-02, 5.3882e-04, 9.8490e-01, 5.3882e-04, 1.9521e-02,
        9.9373e-01, 6.4797e-03, 9.1148e-03, 9.9776e-01, 9.1148e-03, 6.4797e-03,
        9.8194e-01, 7.7850e-03, 9.6305e-03, 9.8316e-01, 9.6305e-03, 1.1036e-02])


In [22]:
print("user, item, label, score")
for user, item, label, score in zip(users, items, labels, probs):
  print(user.item(), item.item(), label.item(), score.item())

user, item, label, score
0 0 1.0 0.9917408227920532
0 4 0.0 0.010929402895271778
0 2 0.0 0.0005388244171626866
0 1 1.0 0.984900951385498
0 2 0.0 0.0005388244171626866
0 5 0.0 0.01952078565955162
1 2 1.0 0.9937263131141663
1 1 0.0 0.006479727104306221
1 5 0.0 0.009114769287407398
1 3 1.0 0.9977631568908691
1 5 0.0 0.009114769287407398
1 1 0.0 0.006479727104306221
2 0 1.0 0.9819430708885193
2 3 0.0 0.007785027846693993
2 1 0.0 0.009630467742681503
2 4 1.0 0.9831642508506775
2 1 0.0 0.009630467742681503
2 2 0.0 0.011036348529160023


In [25]:
with torch.no_grad():
  for u, i in test_items.items():
    print(u, i, ratings[u, i], torch.sigmoid(model(torch.tensor([u]), torch.tensor([i]))))

0 3 tensor(1.) tensor([0.0618])
1 4 tensor(1.) tensor([0.4556])
2 5 tensor(1.) tensor([0.8883])


In [26]:
import math

# 특정 아이템이 user에 대한 점수중 몇번째인지 확인하는 함수
def ndcg_to_k(top_items, target_item, k):
  # top_items에서 target_item의 순서를 잡아서 ndcg 계산해주기
  top_items = top_items[:k].tolist()

  # 없으면 반환해주기
  if target_item not in top_items:
    return 0.0

  # 있으면 ndcg 계산해주기
  rank = top_items.index(target_item) + 1

  return 1 / math.log2(rank+1)

In [29]:
with torch.no_grad():
  for u, i in test_items.items():
    scores = model(
      torch.tensor([u], dtype=torch.float32),
      torch.tensor(ratings[u], dtype=torch.float32)
    )

    print(torch.topk(scores, k=5))


C:\Users\hurwa\AppData\Local\Temp\ipykernel_38132\1979874427.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(ratings[u], dtype=torch.float32)


RuntimeError: Expected tensor for argument #1 'indices' to have one of the following scalar types: Long, Int; but got torch.FloatTensor instead (while checking arguments for embedding)